In [19]:
import pandas as pd

df = pd.read_csv("../data/processed/audible_clean.csv")

df_advance = pd.read_csv(
    "../data/processed/audible_advanced_clean.csv"
)

print("Dataset 1 Shape:", df.shape)
print("Dataset 2 Shape:", df_advance.shape)

Dataset 1 Shape: (5439, 5)
Dataset 2 Shape: (4293, 9)


In [20]:
print("Dataset 1 Columns:")
print(df.columns.tolist())

print("\nDataset 2 Columns:")
print(df_advance.columns.tolist())

Dataset 1 Columns:
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price']

Dataset 2 Columns:
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price', 'Description', 'Listening Time', 'Ranks and Genre', 'Listening_Time_Minutes']


In [21]:
df_advance[[
    "Book Name",
    "Author",
    "Description",
    "Ranks and Genre"
]].head(5)

,Book Name,Author,Description,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,"Over the past three years, Jay Shetty has beco...",",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,Brought to you by Penguin.,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,"In this generation-defining self-help guide, a...",",#3 in Audible Audiobooks & Originals (See Top..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,Brought to you by Penguin.,",#5 in Audible Audiobooks & Originals (See Top..."
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,"Stop going through life, Start growing throug...",",#6 in Audible Audiobooks & Originals (See Top..."


In [22]:
# Display complete genre/rank information for 10 books
for value in df_advance["Ranks and Genre"].head(10):
    print(value)
    print("-" * 80)

,#1 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Personal Success,#1 in Stress Management,#2 in Society & Culture (Books)
--------------------------------------------------------------------------------
,#2 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Meditation (Audible Audiobooks & Originals),#1 in Self-Esteem,#2 in Personal Success
--------------------------------------------------------------------------------
,#3 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#3 in Personal Success,#16 in Personal Development & Self-Help
--------------------------------------------------------------------------------
,#5 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Psychology (Audible Audiobooks & Originals),#3 in Stress Management,#4 in Personal Success
--------------------------------------------------------------------------------
,#

In [23]:
import re

def extract_genres(rank_genre):
    if pd.isna(rank_genre):
        return []

    genres = re.findall(r"#\d+\s+in\s+([^,]+)", rank_genre)

    # Remove the overall Audible ranking
    genres = [
        genre.strip()
        for genre in genres
        if "Audible Audiobooks & Originals" not in genre
    ]

    return genres

df_advance["Genres"] = df_advance["Ranks and Genre"].apply(extract_genres)

df_advance[["Book Name", "Genres"]].head(10)

,Book Name,Genres
0,Think Like a Monk: The Secret of How to Harnes...,"[Personal Success, Stress Management, Society ..."
1,Ikigai: The Japanese Secret to a Long and Happ...,"[Self-Esteem, Personal Success]"
2,The Subtle Art of Not Giving a F*ck: A Counter...,"[Personal Success, Personal Development & Self..."
3,Atomic Habits: An Easy and Proven Way to Build...,"[Stress Management, Personal Success]"
4,Life's Amazing Secrets: How to Find Balance an...,"[Literary Essays, Essays (Books), Spiritualism]"
5,Sapiens,"[History of Civilization, Anthropology (Books)]"
6,Extraordinary Leadership,"[Leadership, Business Careers]"
7,The Intelligent Investor Rev Ed.,"[Personal Finance, Investing & Trading, Analys..."
8,Rich Dad Poor Dad: What the Rich Teach Their K...,"[Personal Finance, Investing & Trading, Person..."
9,The 5AM Club: Own Your Morning. Elevate Your L...,"[Sleep Disorders, Personal Success, Healthy Li..."


In [24]:
# Combine Description and Genres
df_advance["Content"] = (
    df_advance["Description"].fillna("") + " " +
    df_advance["Genres"].apply(lambda x: " ".join(x))
)

# Display the combined content
df_advance[["Book Name", "Content"]].head(5)

,Book Name,Content
0,Think Like a Monk: The Secret of How to Harnes...,"Over the past three years, Jay Shetty has beco..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Brought to you by Penguin. Self-Esteem Persona...
2,The Subtle Art of Not Giving a F*ck: A Counter...,"In this generation-defining self-help guide, a..."
3,Atomic Habits: An Easy and Proven Way to Build...,Brought to you by Penguin. Stress Management P...
4,Life's Amazing Secrets: How to Find Balance an...,"Stop going through life, Start growing throug..."


In [25]:
#TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words="english")

# Convert book content into TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(df_advance["Content"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (4293, 14732)


In [26]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

print("Cosine Similarity Matrix Shape: ", cosine_sim.shape)

Cosine Similarity Matrix Shape:  (4293, 4293)


In [35]:

def recommend_book(book_name, cosine_sim=cosine_sim):

    # Map book titles to positional indices
    indices = pd.Series(
        range(len(df_advance)),
        index=df_advance["Book Name"]
    ).drop_duplicates()

    # Handle invalid book titles
    if book_name not in indices:
        print("Book not found in the dataset.")
        return

    # Get correct book position
    idx = indices[book_name]

    # Get similarity scores
    similarity_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity score
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Exclude selected book itself
    similarity_scores = similarity_scores[1:6]

    # Get recommended book indices
    book_indices = [i[0] for i in similarity_scores]

    # Return recommendations
    return df_advance.iloc[book_indices][
        ["Book Name", "Author", "Rating", "Genres"]
    ]


In [36]:

# Test the recommendation system

book_name = df_advance["Book Name"].iloc[0]
print("Selected Book:", book_name)
recommendations = recommend_book(book_name)

print("\nTop 5 Recommended Books:")
display(recommendations)


Selected Book: Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now

Top 5 Recommended Books:


,Book Name,Author,Rating,Genres
1865,"The Classic Tales Podcast, Season Five",Agatha Christie,NaN,"[Caribbean & Latin American Literature, English]"
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,"[Stress Management, Personal Success]"
1295,The Facebook Effect: The Inside Story of the C...,David Kirkpatrick,4.4,"[Social Media, Technology & Society, E-Commerce]"
448,Ten Arguments For Deleting Your Social Media A...,Jaron Lanier,4.4,"[Social Media, Popular Culture Studies, Techno..."
1043,Social Media Marketing Workbook 2020,Jason McDonald PhD,4.5,"[Social Media, Internet & Social Media (Books)]"


In [37]:

# Test invalid book title

recommendations = recommend_book("Unknown Book XYZ")


Book not found in the dataset.


Evaluation of Recommandation systen

In [38]:

# Evaluate Top 5 recommendations using Cosine Similarity

book_name = df_advance["Book Name"].iloc[0]

indices = pd.Series(
    df_advance.index,
    index=df_advance["Book Name"]
).drop_duplicates()

idx = indices[book_name]

similarity_scores = list(enumerate(cosine_sim[idx]))

similarity_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

similarity_scores = similarity_scores[1:6]

book_indices = [i[0] for i in similarity_scores]
scores = [i[1] for i in similarity_scores]

evaluation_results = df_advance.iloc[book_indices][
    ["Book Name", "Author", "Rating", "Genres"]
].copy()

evaluation_results["Similarity Score"] = scores

print("Selected Book:", book_name)

display(evaluation_results)


Selected Book: Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now


,Book Name,Author,Rating,Genres,Similarity Score
1865,"The Classic Tales Podcast, Season Five",Agatha Christie,NaN,"[Caribbean & Latin American Literature, English]",0.158788
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,"[Stress Management, Personal Success]",0.151893
1295,The Facebook Effect: The Inside Story of the C...,David Kirkpatrick,4.4,"[Social Media, Technology & Society, E-Commerce]",0.143922
448,Ten Arguments For Deleting Your Social Media A...,Jaron Lanier,4.4,"[Social Media, Popular Culture Studies, Techno...",0.133815
1043,Social Media Marketing Workbook 2020,Jason McDonald PhD,4.5,"[Social Media, Internet & Social Media (Books)]",0.133168


In [39]:

# Calculate Average Similarity Score

average_similarity = evaluation_results["Similarity Score"].mean()

print("Average Similarity Score:", round(average_similarity, 4))
print("Average Similarity Percentage:", round(average_similarity * 100, 2), "%")


Average Similarity Score: 0.1443
Average Similarity Percentage: 14.43 %


In [40]:

# Select a book with a known title

book_name = "Atomic Habits: An Easy and Proven Way to Build Good Habits and Break Bad Ones"

recommendations = recommend_book(book_name)

display(recommendations)


,Book Name,Author,Rating,Genres
1982,The Stress Solution: The 4 Steps to Reset Your...,Dr Rangan Chatterjee,4.7,[Stress Management]
850,Happy: Why More or Less Everything Is Absolute...,Derren Brown,4.4,[Personal Success]
2236,Surrounded by Idiots: The Four Types of Human ...,Thomas Erikson,4.5,[Personal Success]
2631,The Antidote: Happiness for People Who Can't S...,Oliver Burkeman,4.5,[Personal Success]
50,To Kill a Mockingbird,Harper Lee,4.5,[]


In [41]:

# Calculate Precision@5

relevant_books = 4
k = 5

precision_at_5 = relevant_books / k

print("Precision@5:", round(precision_at_5, 2))
print("Precision@5 Percentage:", round(precision_at_5 * 100, 2), "%")


Precision@5: 0.8
Precision@5 Percentage: 80.0 %


In [43]:
# Test recommendations for multiple audiobooks

test_books = df_advance["Book Name"].dropna().drop_duplicates().sample(
    5, random_state=42
)

for book in test_books:
    print("\nSelected Book:", book)
    print("-" * 80)

    recommendations = recommend_book(book)

    display(recommendations)


Selected Book: The Mixer: The Story of Premier League Tactics, from Route One to False Nines
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres
4019,The Soccer Mystery: The Boxcar Children Myster...,Gertrude Chandler Warner,5.0,[]
784,"Coaching for Performance, Fourth Edition: GROW...",John Whitmore,4.5,"[Mentoring & Coaching, Leadership]"
1967,Learn Hindi - Level 1: Introduction to Hindi: ...,Innovative Language Learning LLC,NaN,[]
1251,The Third Door: The Wild Quest to Uncover How ...,Alex Banayan,4.7,"[Mentoring & Coaching, Personal Success in Bus..."
3179,"The Science Fiction Hall of Fame, Vol. 2-B",Ben Bova,4.7,[]



Selected Book: Leadership on the Line (Revised): Staying Alive Through the Dangers of Change
--------------------------------------------------------------------------------


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [44]:

print("Cosine Similarity Shape:", cosine_sim.shape)
print("Data Type:", type(cosine_sim))
print("Selected Row Shape:", cosine_sim[0].shape)


Cosine Similarity Shape: (4293, 4293)
Data Type: <class 'numpy.ndarray'>
Selected Row Shape: (4293,)
